#### 1. Read data from the sales_sample.csv file and analyse to identify problem

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType,FloatType
sales_shema = StructType([
  StructField('id',IntegerType()),
  StructField('Name',StringType()),
  StructField('dop',StringType()),
  StructField('phone',LongType()),
  StructField('Amount',LongType()),
  StructField('discount',StringType())]
)

In [0]:
sales_sample_raw_df = (
  spark.read
  .format('CSV')
  .option('header', True)
  .schema(sales_shema)
  .load(path= '/Volumes/dev/spark_db/datasets/spark_programming/data/sales_sample.csv')
)

In [0]:
sales_sample_raw_df.display()

#### CONVERT ID from integer to string and rename it as Transction_id

In [0]:
from pyspark.sql.functions import col
datatype_df= (
    sales_sample_raw_df
    .withColumns({'id': col("id").cast("String")})
)
                 

In [0]:
datatype_df.display()

#### Rename the column names

In [0]:
df = (
  df.withColumnRenamed("id", "Transaction_id") \
    .withColumnRenamed("Name", "customer_name") \
    .withColumnRenamed("phone","customer_phone") \
    .withColumnRenamed("dop", "date_of_purchase") \
      .withColumnRenamed("Amount", "purchase_amount") \
)

%md
#### converted dop to date datatype

In [0]:
from pyspark.sql.functions import expr

df = (
    datatype_df
    .withColumn(
        "date_of_purchase",
        expr("""
        coalesce(
            try_to_date(date_of_purchase,'yyyy-MM-dd'),
            try_to_date(date_of_purchase,'yyyy-MM-d'),
            try_to_date(date_of_purchase,'dd-MM-yyyy')
        )
        """)
    )
)

#### converted  discount string to double, 
note: you cant convert string datatype to float or double
example: 'nil' to cant convert, if it is numberic to you can 
so here you use CASE or WHEN conditional statement in the Syntax
 

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumns({
    "discount":
    when(col("discount") == "nil", 0).otherwise(col("discount").cast('double')),
    "customer_phone": when(col("customer_phone") == "null",0).otherwise(col("customer_phone")) }
)

In [0]:
df=df.withColumn("discount", col("discount").cast("float"))